In [1]:
import pandas as pd 
import numpy as np 
from scipy.stats import percentileofscore

### User Input

In [ ]:
depth_num = 51 #Interval in m/ft
interval_moving = 1
InputExcelFile = "NWX-1.xlsx"
OutputExcelFile = "NWX-1_out.xlsx"

### Intensity Calculation Code

In [10]:
WellLog_DF = pd.read_excel(InputExcelFile, sheet_name="Data")

WellLog_DF['IntensityIndex'] = 999

Start_Idx_Array = np.arange(0,len(WellLog_DF)-depth_num, interval_moving)
Mid_Idx_Array = np.arange(0,len(WellLog_DF)-depth_num, interval_moving)+int(depth_num/2)
End_Idx_Array = np.arange(0,len(WellLog_DF)-depth_num, interval_moving)+int(depth_num)

i = 0
for Start_Idx,Mid_Idx, End_Idx in zip(Start_Idx_Array,Mid_Idx_Array, End_Idx_Array):

    WellLog_DF_Slice = WellLog_DF.loc[Start_Idx:End_Idx,['Depth','Log']]
    WellLog_DF_Slice_unique = WellLog_DF_Slice.groupby((WellLog_DF_Slice['Log'] != WellLog_DF_Slice['Log'].shift(1)).cumsum()).mean()

    weight_temp = 120-np.abs(percentileofscore(WellLog_DF_Slice['Depth'].values, WellLog_DF_Slice_unique['Depth'].tolist(), kind='mean')-50)
    weight_temp = np.clip(weight_temp,1,100)

    IntensityValues = np.sum(weight_temp * (WellLog_DF_Slice_unique['Log'].values))/100
    # print(IntensityValues)
    WellLog_DF.loc[Mid_Idx,'IntensityIndex'] = IntensityValues

WellLog_DF.to_excel(OutputExcelFile)